# Assignment 05: Clean a Lab Export

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [1]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

DATA_PATH = Path("data") / "lab_visits.csv"
if not DATA_PATH.exists():
    subprocess.run([sys.executable, "generate_data.py"], check=True)
manifest = json.loads(Path("data_manifest.json").read_text())
for name, expected in manifest["files"].items():
    digest = hashlib.sha256(Path(name).read_bytes()).hexdigest()
    assert digest == expected["sha256"], f"{name} differs from the supplied data: delete data/ and run this cell again"
    print(name, "matches data_manifest.json")

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
MISSING_PATH = OUTPUT_DIR / "missing_report.csv"
PROFILE_PATH = OUTPUT_DIR / "typed_profile.csv"
CATEGORY_PATH = OUTPUT_DIR / "category_counts.csv"
VALIDATION_PATH = OUTPUT_DIR / "validation_report.csv"
ROW_COUNTS_PATH = OUTPUT_DIR / "row_counts.csv"
SUMMARY_PATH = OUTPUT_DIR / "clean_summary.csv"

NumPy: 2.3.3
pandas: 3.0.5
data/lab_visits.csv matches data_manifest.json


## Task 1: Find missing values and repeats

### 1.1 Count missing values

Run this cell as it is: it reads the export with no options.

In [2]:
labs = pd.read_csv(DATA_PATH)
rows_read = len(labs)
print("shape:", labs.shape)
print(labs.dtypes)
labs.head()

shape: (60918, 8)
patient_id           str
visit_date           str
clinic               str
gender               str
age                  str
weight_kg        float64
hba1c_pct        float64
glucose_mg_dl    float64
dtype: object


,patient_id,visit_date,clinic,gender,age,weight_kg,hba1c_pct,glucose_mg_dl
0,D10425,2024-01-02,Bayview,M,64,122.7,6.7,164.0
1,D10810,2024-01-02,Mission,F,74,42.0,7.7,172.0
2,D11509,2024-01-02,SUNSET,F,57,70.0,7.0,178.0
3,D13213,2024-01-02,Sunset,F,51,62.3,7.1,172.0
4,D13220,2024-01-02,Bayview,male,61,125.4,5.6,104.0


> **Checkpoint: `output/missing_report.csv`**
> Eight rows under the header `column,missing_count,missing_pct`.

In [3]:
# TODO: a DataFrame with the columns missing_count and missing_pct, one row per column of labs
# TODO: name the index column

counts = labs.isna().sum()

pcts = (labs.isna().mean() * 100).round(1)

missing_report = pd.DataFrame({
    "missing_count": counts,
    "missing_pct": pcts
})

missing_report.index.name = "column"

display(missing_report)
# TODO: save missing_report to MISSING_PATH, keeping its index

missing_report.to_csv(MISSING_PATH)

,missing_count,missing_pct
column,,
patient_id,0,0.0
visit_date,196,0.3
clinic,248,0.4
gender,117,0.2
age,93,0.2
weight_kg,364,0.6
hba1c_pct,4882,8.0
glucose_mg_dl,3741,6.1


### 1.2 Drop no-lab visits and exact repeats

In [4]:
# TODO: drop the rows where both hba1c_pct and glucose_mg_dl are missing, and assign the result back to labs
labs = labs.dropna(subset=["hba1c_pct", "glucose_mg_dl"], how="all")

rows_with_labs = len(labs)
print("rows with labs:", rows_with_labs)

# TODO: the number of rows identical to an earlier row
exact_repeats = labs.duplicated().sum()  
print("exact repeats:", exact_repeats)

# TODO: drop the exact repeats and assign the result back to labs
labs = labs.drop_duplicates()

rows_unique = len(labs)
print("unique rows:", rows_unique)

rows with labs: 59977
exact repeats: 491
unique rows: 59486


## Task 2: Replace codes and convert types

> **Checkpoint: `output/typed_profile.csv`**
> Five rows under the header `column,dtype,missing_count,min,max,mean`.

In [ ]:
# TODO: replace -999 with np.nan in weight_kg and glucose_mg_dl only, and assign the result back to labs
labs = labs.replace({"weight_kg": {-999: np.nan},
              "glucose_mg_dl": {-999: np.nan}})


# TODO: convert age to Int64, with unknown as missing
labs["age"] = pd.to_numeric(labs["age"], errors="coerce").astype("Int64")

# TODO: convert visit_date to dates, with impossible dates as NaT
labs["visit_date"] = pd.to_datetime(labs["visit_date"], 
                                    format="%Y-%m-%d", 
                                    errors="coerce")
print(labs.dtypes)

typed = labs[["visit_date", "age", "weight_kg", "hba1c_pct", "glucose_mg_dl"]]
# TODO: a DataFrame of dtype, missing_count, min, max, and mean for each column of typed
dtypes = typed.dtypes.astype(str)
missing_counts =  typed.isna().sum()
mins = typed.min()
maxes =  typed.max()
means = typed.mean(numeric_only=True).round(2)

typed_profile = pd.DataFrame({
    "dtype": dtypes,
    "missing_count": missing_counts,
    "min": mins,
    "max": maxes,
    "mean": means
})

# TODO: name the index column
typed_profile.index.name = "column"

display(typed_profile)

# TODO: save typed_profile to PROFILE_PATH, keeping its index
typed_profile.to_csv(PROFILE_PATH)

patient_id                  str
visit_date       datetime64[us]
clinic                      str
gender                      str
age                       Int64
weight_kg               float64
hba1c_pct               float64
glucose_mg_dl           float64
dtype: object


,dtype,missing_count,min,max,mean
column,,,,,
age,Int64,236,21,94,58.59
glucose_mg_dl,float64,3226,55.0,407.0,145.03
hba1c_pct,float64,3907,3.9,99.0,6.77
visit_date,datetime64[us],299,2024-01-02 00:00:00,2026-06-30 00:00:00,<NA>
weight_kg,float64,940,29.9,1258.0,87.43


## Task 3: Standardize text and encode categories

### 3.1 Clean the text columns

In [6]:
display(labs["clinic"].value_counts(dropna=False))

# TODO: strip and upper-case patient_id
# TODO: strip and title-case clinic, then fill its blanks with "Unknown"
# TODO: map every spelling of gender to F, M, or X with a dictionary

clinic
Mission       18539
Sunset        13976
Excelsior     11739
Bayview        7846
mission        1011
sunset          831
excelsior       681
MISSION         626
SUNSET          445
EXCELSIOR       437
bayview         431
 Mission        417
Mission         410
 Sunset         348
Sunset          337
 Excelsior      290
Excelsior       265
BAYVIEW         260
NaN             238
 Bayview        188
Bayview         170
mission           1
Name: count, dtype: int64

### 3.2 Bin HbA1c and count the categories

> **Checkpoint: `output/category_counts.csv`**
> Eleven rows under the header `indicator,visits`, from `clinic_Bayview` to `a1c_category_diabetes`.

In [7]:
# TODO: add a1c_category, the HbA1c band of each visit

coded = None  # TODO: one 0/1 column per category of clinic, gender, and a1c_category

category_counts = None  # TODO: the visits per indicator column of coded
# TODO: name the index indicator and the Series visits

display(category_counts)
# TODO: save category_counts to CATEGORY_PATH, keeping its index

None

## Task 4: Flag outliers and count rule failures

Audit flags identify values to review. Missing gender is flagged but allowed in the cleaned table.

> **Checkpoint: `output/validation_report.csv`**
> Six rows under the header `rule,failing_rows`.

In [8]:
q1 = None  # TODO: glucose quartiles
q3 = None
iqr = None
lower = None  # TODO: the IQR fences
upper = None
print(f"Q1 {q1}, Q3 {q3}, fences {lower} to {upper}")

# TODO: add glucose_outlier, True where glucose_mg_dl is below lower or above upper

rules = None  # TODO: a DataFrame of True/False with the six rule columns README.md lists

validation_report = None  # TODO: the number of failing rows per rule
# TODO: name the index rule and the Series failing_rows

display(validation_report)
# TODO: save validation_report to VALIDATION_PATH, keeping its index

Q1 None, Q3 None, fences None to None


None

## Task 5: Clean and summarize

### 5.1 Apply the decisions and count the rows

> **Checkpoint: `output/row_counts.csv`**
> Six rows under the header `step,rows`, from `rows_read` to `weight_blanked`.

In [9]:
clean = None  # TODO: a copy of labs
# TODO: keep the rows that pass both patient_id_format and visit_date_present

hba1c_outside = None  # TODO: the number of HbA1c values below 4 or above 20 in clean
weight_outside = None  # TODO: the number of weights below 30 or above 250 in clean
# TODO: blank the HbA1c and weight values outside their ranges
# TODO: recompute clean['a1c_category'] from the cleaned HbA1c using Task 3.2's bins and labels
# Keep category_counts.csv as the Task 3 snapshot
# TODO: reset the index, dropping the old one

row_counts = None  # TODO: a Series of the three row counts and the three decisions README.md lists, named rows
# TODO: name the index step

print("shape:", clean.shape)
display(row_counts)
# TODO: save row_counts to ROW_COUNTS_PATH, keeping its index

AttributeError: 'NoneType' object has no attribute 'shape'

### 5.2 Validate, then save the summary

> **Checkpoint: `output/clean_summary.csv`**
> Four rows under the header `,count,mean,std,min,25%,50%,75%,max`.

In [ ]:
checks = None  # TODO: a Series of the five contract checks README.md lists
display(checks)
# TODO: assert that every check passes, showing the failing ones if not

clean_summary = None  # TODO: the summary statistics of the four number columns of clean, one row per column
display(clean_summary)
# TODO: save clean_summary to SUMMARY_PATH, keeping its index

### 5.3 Save the cleaned table as Parquet

> **Checkpoint: `data/labs_clean.parquet`**
> Not committed or checked: `data/` stays out of Git.

In [ ]:
parquet_path = Path("data") / "labs_clean.parquet"
# TODO: save clean to parquet_path

clean_back = None  # TODO: parquet_path read back
print(clean_back.dtypes)
print("same as clean:", clean_back.equals(clean))

## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [ ]:
from check_assignment import run_checks

_ = run_checks()